# Market Regime Detection with Network and Volatility Structure\n\nThis notebook studies how relationships between market assets change across **calm** and **stressed** periods.\n\nWe will:\n- download price data for a basket of ETFs\n- compute rolling returns, volatility, and correlations\n- turn correlations into network graphs\n- track when the market becomes more tightly connected\n- compare calm vs stressed regimes\n\nThis is a strong finance + complexity project because it connects:\n- **regime shifts**\n- **dynamic networks**\n- **contagion structure**\n- **diversification breakdown**\n

## 1. Imports\n\nWe import Python packages and the helper functions from our `src` module.

In [ ]:
import warnings\nwarnings.filterwarnings('ignore')\n\nimport pandas as pd\nimport matplotlib.pyplot as plt\nimport plotly.graph_objects as go\nfrom plotly.subplots import make_subplots\n\nfrom src.regime_analysis import (\n    DEFAULT_TICKERS,\n    download_prices,\n    compute_returns,\n    rolling_annualized_volatility,\n    compute_rolling_market_metrics,\n    label_regimes,\n    get_regime_snapshot_dates,\n    correlation_snapshot,\n    plot_network,\n    compare_regime_summary,\n    diversification_writeup,\n)\n\nplt.rcParams['figure.figsize'] = (12, 6)\npd.set_option('display.max_columns', 50)

## 2. Choose assets and date range\n\nWe use ETFs by default because they are easier to interpret than individual stocks for a first version of this project.

In [ ]:
tickers = DEFAULT_TICKERS\nstart_date = '2018-01-01'\ncorr_window = 63      # about 3 months of trading days\nvol_window = 21       # about 1 month of trading days\ncorr_threshold = 0.60\n\ntickers

## 3. Download price data\n\nWe download adjusted prices and inspect the first few rows.

In [ ]:
prices = download_prices(tickers=tickers, start=start_date)\nprices.head()

## 4. Compute daily returns and rolling volatility

In [ ]:
returns = compute_returns(prices)\nrolling_vol = rolling_annualized_volatility(returns, window=vol_window)\n\nprint('Returns shape:', returns.shape)\nrolling_vol.tail()

## 5. Compute rolling market structure metrics\n\nFor each rolling window, we calculate:\n- average volatility\n- average pairwise correlation\n- share of strong correlation links\n- network density\n- clustering\n

In [ ]:
metrics = compute_rolling_market_metrics(\n    returns,\n    window=corr_window,\n    corr_threshold=corr_threshold,\n)\n\nlabeled_metrics = label_regimes(metrics, stress_percentile=0.80)\nlabeled_metrics.head()

## 6. Interactive rolling dashboard\n\nThis dashboard helps us see when the market becomes more connected and more volatile.

In [ ]:
fig = make_subplots(\n    rows=4, cols=1, shared_xaxes=True, vertical_spacing=0.04,\n    subplot_titles=(\n        'Mean Volatility',\n        'Average Pairwise Correlation',\n        'Strong Correlation Share',\n        'Stress Score'\n    )\n)\n\nfig.add_trace(go.Scatter(x=labeled_metrics.index, y=labeled_metrics['mean_volatility'], name='Mean Volatility'), row=1, col=1)\nfig.add_trace(go.Scatter(x=labeled_metrics.index, y=labeled_metrics['avg_pairwise_corr'], name='Avg Correlation'), row=2, col=1)\nfig.add_trace(go.Scatter(x=labeled_metrics.index, y=labeled_metrics['strong_corr_share'], name='Strong Corr Share'), row=3, col=1)\nfig.add_trace(go.Scatter(x=labeled_metrics.index, y=labeled_metrics['stress_score'], name='Stress Score'), row=4, col=1)\n\nstress_dates = labeled_metrics.index[labeled_metrics['regime'] == 'stressed']\nfor date in stress_dates[::max(1, len(stress_dates)//15)]:\n    fig.add_vline(x=date, line_width=1, opacity=0.20)\n\nfig.update_layout(height=900, title='Rolling Market Structure Dashboard', showlegend=False)\nfig.show()

## 7. Pick one calm snapshot and one stressed snapshot\n\nWe choose the dates with the lowest and highest stress score.

In [ ]:
calm_date, stressed_date = get_regime_snapshot_dates(labeled_metrics)\ncalm_date, stressed_date

## 8. Build correlation snapshots for those two dates

In [ ]:
corr_calm = correlation_snapshot(returns, calm_date, window=corr_window)\ncorr_stressed = correlation_snapshot(returns, stressed_date, window=corr_window)\n\ncorr_calm.head()

## 9. Network plots\n\nThese plots are the heart of the project.\n\nIn stressed periods, you will often see:\n- more edges\n- tighter clustering\n- fewer isolated pockets\n\nThat means the market is behaving more like one connected system.

In [ ]:
plot_network(corr_calm, threshold=corr_threshold, title=f'Calm Network: {calm_date.date()}')\nplt.show()\n\nplot_network(corr_stressed, threshold=corr_threshold, title=f'Stressed Network: {stressed_date.date()}')\nplt.show()

## 10. Compare summary statistics by regime

In [ ]:
summary = compare_regime_summary(labeled_metrics)\nsummary

## 11. Short write-up: When diversification weakens

In [ ]:
writeup = diversification_writeup(summary)\nprint(writeup)

## 12. Interpretation guide\n\nQuestions to answer in your own words:\n\n1. When does average correlation spike?\n2. Do volatility spikes and correlation spikes happen together?\n3. Does the stressed network look denser than the calm one?\n4. What does that imply for diversification?\n5. Which ETFs seem most central in the stressed network?\n\nA good headline for the write-up is:\n\n> **When diversification weakens: assets that usually look distinct begin moving together during stress.**\n

## 13. Good next extensions\n\n- replace ETFs with 30-50 stocks\n- add community detection\n- compare 2008, 2020, and 2022 windows\n- try a different network edge threshold\n- build a minimum spanning tree\n- use a formal regime model such as an HMM\n